# motivation

## Preprogress

In [ ]:
import pandas as pd
import glob
import numpy as np
import re
import os


folder_path = 'Path/to/inference'
folder_name = "inference_folder"
model_name = folder_name.split('_')[-1]
kind_exp = folder_name.split('_')[-2]

file_paths = glob.glob(folder_path + folder_name + "/result_index_*.csv")

def extract_index(path):
    match = re.search(r"result_index_(\d+)_", os.path.basename(path))
    return int(match.group(1)) if match else 1e9   #
file_paths = sorted(file_paths, key=extract_index)


merged_df = pd.concat([pd.read_csv(fp) for fp in file_paths], ignore_index=True)

profile_columns = merged_df.columns[:5]
res_columns = merged_df.columns[list(range(19,45,2))]
human_columns = [str(i) for i in range(6,19)]

print(merged_df.shape) 
merged_df.head(2)


In [ ]:
def process_text_result(text):
    if text is None:
        return "", ""
    text = str(text)
    
    
    think_match = re.search(r'(.*?)</think>', text, re.DOTALL)
    thinking_answer = think_match.group(1) if think_match else ""
    

    text_without_think = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    
    filt_str = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9=()（）]', '', text_without_think)

    match = re.search(r'选择(.*?)$', filt_str)
    if match:
        choice = match.group(1)
    else:
        choice = ''
    
    return choice, thinking_answer

import numpy as np
import json

choice_columns,think_columns = [],[]
for question_id, question_column in enumerate(res_columns):
    new_col_name = str(question_id)+'_chocice'
    choice_columns.append(new_col_name)   
    new_text_name = str(question_id)+'_reason_clean'   
    think_columns.append(new_text_name)
    for index,text in enumerate(merged_df[question_column]):
        choice, thinking_answer = process_text_result(text)
        merged_df.at[index,new_col_name] = choice
        merged_df.at[index,new_text_name] = thinking_answer


In [ ]:
import numpy as np
import json

question_path = './Code/Data/wvs_quesitons.json'
with open(question_path, 'r', encoding='utf-8') as file:
    question_dict = json.load(file)

question_list = []
for question_index,question_value in question_dict.items():
    question_list.append(question_value)

choice_num_columns = [str(i)+'_choice_num' for i in range(13)]
for question_id,question in enumerate(question_list):
    option_len = len(question['option'])
    option_to_num = {option: idx for idx, option in enumerate(question['option'])}
    

    def custom_map(value):
        if not isinstance(value, str): 
            return value  
        for key, num in option_to_num.items():
            if key == value:
                return num
        return np.nan  
 
    merged_df[choice_num_columns[question_id]] = merged_df[choice_columns[question_id]].apply(custom_map)
   
    merged_df[choice_num_columns[question_id]] = merged_df[choice_num_columns[question_id]] / (option_len - 1)
    
    merged_df[human_columns[question_id]] = merged_df[human_columns[question_id]].apply(custom_map)
    merged_df[human_columns[question_id]] = merged_df[human_columns[question_id]] / (option_len - 1)


    


In [ ]:

profile_dict = {
    '1': {'女性': 0, '男性': 1},
    '2': {'学前教育/无学历': 0, '小学': 0, '初中': 1, '高中/中专': 2, '大学本科': 3, '硕士': 3}, # 47 62 42 49
    '3': {'第1级 (最低)': 0, '第2级': 0, '第3级': 2, '第4级': 2, '第5级': 3, '第6级': 4, '第7级': 4, '第8级': 4, '第9级': 4}, # 47 65 41 47
    '4': {'从未工作过': 0, 
          '农业劳动者（如：农场劳工、拖拉机驾驶员）': 1, '非技术工人（如：劳工、搬运工、非技术工厂工人、清洁工）': 1,
          '半技术工人（如：泥瓦匠、公交车司机、罐头厂工人、木工、钣金工、面包师）': 2, '技术工人（如：工头、汽车修理工、印刷工、女缝纫工、模具工、电工）': 2,
           '销售人员（如：销售经理、店主、店员、保险代理人、采购员）': 3, '服务人员（如：餐厅老板、警察、服务员、理发师、看管员）': 3, 
           '专业技术人员（如：医生、教师、工程师、艺术家、会计师、护士）': 4,'文职人员（如：秘书、办事员、办公室经理、公务员、记账员）': 4,
          '农场主、农场经理': 4, '高级管理人员（如：银行家、大企业高管、高级政府官员、工会官员）': 4}, #40 28 35 47 50
    '0': None, #39 43 50 34 34
    }

age_values = sorted(merged_df[profile_columns[0]].dropna().unique())
age_bins = np.array_split(age_values, 5) # 18-29 30-39 40-49 50-60 61-70
age_mapping = {}
for bin_idx, bin_vals in enumerate(age_bins):
    for val in bin_vals:
        age_mapping[int(val)] = bin_idx


for idx, profile in enumerate(profile_columns):
    if idx == 0:  # 年
        merged_df[profile] = merged_df[profile].apply(lambda x: age_mapping.get(int(x), np.nan) if pd.notna(x) else np.nan)
        # print(merged_df[profile].value_counts())
    elif profile_dict[profile]: 
        merged_df[profile] = merged_df[profile].map(profile_dict[profile])

merged_df.to_csv(folder_path + folder_name + "/merged_choicenum_result.csv", index=False)
merged_df.columns,merged_df.head(2)


## cal matrix

In [ ]:
import json
import numpy as np

question_path = './Code/Data/wvs_questions.json'
with open(question_path, 'r', encoding='utf-8') as file:
    question_dict = json.load(file)
    
question_list = []
for topic,topic_value in question_dict.items():
    for questin_id,question_value in topic_value.items():
        question_list.append(question_value['question'])

option_num_list = [len(list(item['option'])) for item in question_list]
max_entropy_list = [np.log2(n) for n in option_num_list]

all_possible_values_list = []

for question_id,question in enumerate(question_list):
    option_len = len(question['option'])
    all_possible_values_list.append([i/(option_len-1) for i in range(option_len)])

In [ ]:
import pandas as pd
import numpy as np
def load_data(result_path):
    profile_columns = ['0','1','2','3','4']
    df = pd.read_csv(result_path+'/merged_choicenum_result.csv')
    for col in profile_columns:
        df[col] = df[col].fillna(-1).astype(int) 
    return df

In [ ]:
import numpy as np 

def calculate_entropy(series):
    series = series.dropna()
    counts = series.value_counts()
    probabilities = counts / counts.sum()
    entropy = -np.sum(probabilities * np.log2(probabilities))
    if entropy==-0.00:
        entropy = 0
    return entropy

def calculate_rmse(series,human_series):
    if len(series) != len(human_series):
        raise ValueError("different length")
    mask = ~np.isnan(series) & ~np.isnan(human_series) 
    if not np.any(mask):  
        print("no valid data")
        return np.nan
    
    # 应用掩码，过滤掉包含NaN的行
    series = series[mask]
    human_series = human_series[mask]

    squared_diff = (series - human_series) ** 2
    rmse = np.sqrt(np.mean(squared_diff))
    return rmse


from scipy.stats import wasserstein_distance
import numpy as np
def calculate_wasserstein_grouped(df_true, df_pred,
                                   feat_col, true_col, pred_col,
                                   all_possible_values=None, eps=1e-8):
   
    wasserstein_feat = []
    for f_col in feat_col:
        tmp = pd.DataFrame({
            'f': df_true[f_col],
            'true': df_true[true_col],
            'pred': df_pred[pred_col]
        })
        wasserstein_list = []
   
        for f_val, sub in tmp.groupby('f', observed=True):
         
            mask = ~sub['pred'].isna()
            true_vals = sub.loc[mask, 'true'].values
            pred_vals = sub.loc[mask, 'pred'].values
            if len(true_vals) == 0 or len(pred_vals) == 0:
                wasserstein_list.append(np.nan)
                continue
     
            w_dist = wasserstein_distance(true_vals, pred_vals)
         
            wasserstein_list.append(1 - w_dist)
        wasserstein_feat.append(np.nanmean(wasserstein_list))
    return np.nanmean(wasserstein_feat) if wasserstein_feat else np.nan

def Cal_enr_kl_group_test(df,human_df):
 
    choice_columns = [str(i)+'_chocice' for i in range(0,15)]
    human_columns = [str(i)+'_choice_num' for i in range(0,15)]
    result_dict = {'entropies':{},'rmse':{},'wasserstein':{}} 

    
   
    for i,choice_column in enumerate(choice_columns):
    
        temp_en = calculate_entropy(df[choice_column])/max_entropy_list[i]
        result_dict['entropies'][i] = temp_en
    average_en = np.nanmean(list(result_dict['entropies'].values()))
    result_dict['entropies']['average'] = average_en
    selected_values = [result_dict['entropies'][i] for i in [0, 3, 6, 9, 12] if i in result_dict['entropies']]
    average_en_test = np.nanmean(selected_values) 
    result_dict['entropies']['average_test'] = average_en_test

    for i,choice_column in enumerate(choice_columns):
        temp_dif = calculate_rmse(df[choice_column],human_df[human_columns[i]])
        result_dict['rmse'][i] = temp_dif
        temp_dif = calculate_acc(df[choice_column],human_df[human_columns[i]])
        result_dict['acc'][i] = temp_dif
    average_rmse = np.nanmean(list(result_dict['rmse'].values()))
    result_dict['rmse']['average'] = average_rmse
    selected_values = [result_dict['rmse'][i] for i in [0, 3, 6, 9, 12] if i in result_dict['rmse']]
    average_rmse_test = np.nanmean(selected_values) 
    result_dict['rmse']['average_test'] = average_rmse_test

   
    feature_cols = [str(i) for i in range(5)]        # 0 1 2 3 4
    for i, choice_column in enumerate(choice_columns):
        human_column = human_columns[i]
        temp_wasserstein = calculate_wasserstein_grouped(
            df_true=human_df,
            df_pred=df,
            feat_col=feature_cols,
            true_col=human_column,
            pred_col=choice_column,
            all_possible_values=all_possible_values_list[i]
        )
        result_dict['wasserstein'][i] = temp_wasserstein
    
    average_wasserstein = np.nanmean(list(result_dict['wasserstein'].values()))
    result_dict['wasserstein']['average'] = average_wasserstein

    selected_values = [result_dict['wasserstein'][i] for i in [0, 3, 6, 9, 12] if i in result_dict['wasserstein']]
    average_wasserstein_test = np.nanmean(selected_values)
    result_dict['wasserstein']['average_test'] = average_wasserstein_test

    return result_dict

In [ ]:
def save_en_kl(model_name,result_dict,save_path,non_human):
    with open(save_path+'/motivation_result.json', "w", encoding="utf-8") as f:
        json.dump(result_dict, f, ensure_ascii=False, indent=2)
    print(f'model：{model_name} \n en-test:{result_dict["entropies"]["average_test"]:.3f}\n wa-test:{result_dict["wasserstein"]["average_test"]:.3f}\n rmse:{(1-result_dict["rmse"]["average_test"]):.3f}')
    return result_dict


In [ ]:

profile_columns = ['0','1','2','3','4']
choice_columns = [str(i)+'_chocice' for i in range(0,15)]


BASE_FOLDER_PATH= 'Path/to/inference'
BASE_MODEL_NAME= ['inference_folder',] 
for base_model in BASE_MODEL_NAME:
    base_result_path = BASE_FOLDER_PATH+base_model
    base_df = load_data(base_result_path)
    direct_result = Cal_enr_kl_group_test(base_df,True,base_df) 
    result_csv = save_en_kl(base_model,direct_result,base_result_path,True)